# AF5 | Modelo dinámico y simulación de un péndulo robot de 1 GDL

**Objetivo:** simular la posición y la velocidad angular de una articulación accionada por un motor durante 10 s, usando el modelo obtenido con Euler–Lagrange.

**Convención:** el ángulo $q$ se mide desde la vertical inferior, con sentido positivo antihorario. Los estados son $x_1=q$ y $x_2=\dot q$.

> **Nota de autoría:** notebook elaborado para esta actividad a partir de la ecuación dinámica desarrollada en el reporte. Como material de consulta se utilizó el ejemplo público `Python_PenduloRobot_1GdL.ipynb` del repositorio *Modelado_y_Simulacion_de_Sistemas_Mecatronicos* (ver referencias al final). No se atribuye la autoría de ese material a este equipo.

## 1. Modelo matemático

Energías del sistema:

$$T=\frac{1}{2}I_r\dot q^2,\qquad V=mgl_c(1-\cos q)$$

Aplicando Euler–Lagrange e incluyendo par de motor y fricciones:

$$I_r\ddot q+b\dot q+mgl_c\sin(q)+f_c\tanh(100000\dot q)=\tau(t)$$

El par aplicado es $\tau(t)=1.5\sin(t)$ N·m. La tangente hiperbólica aproxima la función signo de la fricción de Coulomb; **no** es un modelo completo de fricción estática.

En espacio de estados:

$$\dot x_1=x_2$$

$$\dot x_2=\frac{1.5\sin(t)-mgl_c\sin(x_1)-bx_2-f_c\tanh(100000x_2)}{I_r}$$

Condiciones iniciales: $x_1(0)=0$ rad, $x_2(0)=0$ rad.

## 2. Bibliotecas y parámetros

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from pathlib import Path

# Parámetros entregados en la actividad (unidades SI)
PARAMETROS = {
    "Ir": 0.16,  # kg m^2
    "m": 5.0,    # kg
    "g": 9.81,   # m/s^2
    "lc": 0.01,  # m
    "b": 0.14,   # N m s/rad
    "fc": 0.45,  # N m
}

TIEMPO_FINAL = 10.0  # segundos
MUESTRAS = 2001      # incluye t=0 y t=10
ESTADO_INICIAL = (0.0, 0.0)  # [posición (rad), velocidad (rad/s)]

print("Parámetros cargados:", PARAMETROS)

## 3. Motor y ecuaciones de estado

In [ ]:
def par_motor(t):
    """Par de entrada en N·m, según el modelo de referencia."""
    return 1.5 * np.sin(t)


def modelo_pendulo(t, estado, parametros):
    """Devuelve [velocidad angular, aceleración angular]."""
    angulo, velocidad = estado
    Ir = parametros["Ir"]

    par_gravitacional = parametros["m"] * parametros["g"] * parametros["lc"] * np.sin(angulo)
    par_viscoso = parametros["b"] * velocidad
    par_coulomb = parametros["fc"] * np.tanh(100000.0 * velocidad)

    aceleracion = (par_motor(t) - par_gravitacional - par_viscoso - par_coulomb) / Ir
    return [velocidad, aceleracion]

# Comprobación de condiciones iniciales del modelo
print("Derivadas iniciales [rad/s, rad/s²]:", modelo_pendulo(0, ESTADO_INICIAL, PARAMETROS))

## 4. Solución numérica durante 10 segundos

Usamos `solve_ivp` con **LSODA**, un integrador que puede manejar transiciones pronunciadas como las introducidas por `tanh(100000 x₂)`. El intervalo y los parámetros se mantienen sin cambios.

In [ ]:
instantes = np.linspace(0.0, TIEMPO_FINAL, MUESTRAS)

solucion = solve_ivp(
    fun=lambda t, x: modelo_pendulo(t, x, PARAMETROS),
    t_span=(0.0, TIEMPO_FINAL),
    y0=ESTADO_INICIAL,
    t_eval=instantes,
    method="LSODA",
    rtol=1e-7,
    atol=1e-9,
)

if not solucion.success:
    raise RuntimeError(f"Falló la integración: {solucion.message}")

segundos = solucion.t
angulo_rad = solucion.y[0]
velocidad_rad_s = solucion.y[1]
angulo_grados = np.rad2deg(angulo_rad)
velocidad_grados_s = np.rad2deg(velocidad_rad_s)

assert len(segundos) == MUESTRAS
assert np.all(np.isfinite(solucion.y))
assert np.allclose(solucion.y[:, 0], ESTADO_INICIAL)

print(f"Simulación completada: {segundos[0]:.1f} a {segundos[-1]:.1f} s")
print(f"Posición final: {angulo_grados[-1]:.2f}°")
print(f"Velocidad final: {velocidad_grados_s[-1]:.2f}°/s")

## 5. Gráficas de resultados

Las dos figuras se guardarán como archivos **PNG** en la misma carpeta desde la que se ejecuta el notebook. Esos archivos pueden incluirse en el reporte.

In [ ]:
carpeta_resultados = Path("resultados_AF5")
carpeta_resultados.mkdir(exist_ok=True)


def graficar_respuesta(t, datos, titulo, eje_y, nombre_archivo, color):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, datos, color=color, linewidth=2)
    ax.set(title=titulo, xlabel="Tiempo (s)", ylabel=eje_y)
    ax.set_xlim(0, TIEMPO_FINAL)
    ax.grid(True, alpha=0.35)
    fig.tight_layout()
    fig.savefig(carpeta_resultados / nombre_archivo, dpi=200, bbox_inches="tight")
    plt.show()


graficar_respuesta(
    segundos, angulo_grados,
    "Posición angular del péndulo-robot",
    "Posición angular (°)", "posicion_angular_AF5.png", "tab:blue"
)

graficar_respuesta(
    segundos, velocidad_grados_s,
    "Velocidad angular del péndulo-robot",
    "Velocidad angular (°/s)", "velocidad_angular_AF5.png", "tab:red"
)

print(f"Imágenes guardadas en: {carpeta_resultados.resolve()}")

## 6. Interpretación de resultados

La posición angular puede superar los 360° porque la gráfica representa el **ángulo acumulado**. Los cambios de signo de la velocidad indican una inversión del sentido de giro. Esta respuesta se obtiene con el par sinusoidal del motor y las contribuciones de gravedad y fricción definidas en el modelo.

Los resultados calculados son una **simulación numérica**, no mediciones experimentales. Si se modifican parámetros o tolerancias, los valores pueden variar ligeramente.

### Referencias de consulta

- Material de la actividad **AF5. Simulación de un sistema a partir de ecuaciones de Euler–Lagrange** (instrucciones proporcionadas por el docente).
- Ejemplo de código consultado: [*Modelado_y_Simulacion_de_Sistemas_Mecatronicos*, sección PenduloRobot_1GdL](https://github.com/dilp90/Modelado_y_Simulacion_de_Sistemas_Mecatronicos/tree/main/PenduloRobot_1GdL). La ecuación y la señal de par del ejemplo se emplearon como referencia; este notebook presenta una implementación y organización propias.